# GRAT walkthrough: Oak Valley sample data

GRAT models how water available for recharge (WAFR) is allocated, day by day, to canals, basins and farm fields, and reports how much of it reaches groundwater.

This notebook runs the full workflow end to end against **Oak Valley**, a synthetic sample dataset that ships with the `grat` package (it's included whether you `pip install grat` or clone the repo) - see [`grat/datasets/OakValley/README.md`](../src/grat/datasets/OakValley/README.md) for what it contains. It is not real hydrologic, crop or parcel data.

Steps:
1. Load the sample data (shapefile + csvs).
2. **Check it for problems before running anything**, with `grat.validate_grat_inputs`.
3. Build the model's site objects.
4. Run the model and produce reports.
5. Read a report back as a dataframe.


## 1. Load the sample data

In [ ]:
from importlib.resources import files
from pathlib import Path

import pandas as pd

OAK_VALLEY = Path(str(files("grat.datasets") / "OakValley"))
list(OAK_VALLEY.iterdir())

`load_sites` reads the attribute table of any vector file GDAL can read (shapefile, file geodatabase, geopackage, geojson) or a plain csv - geometry is never read, so no GIS library beyond `pyogrio` is needed.

The sites layer's raw field names (`GRATID`, `R_TYPE`, ...) rarely match what the model classes expect (`grat_id`, `recharge_type`, ...), so `attribute_renamer` maps one to the other. This is the mapping for Oak Valley's shapefile; bringing your own data means building the equivalent dictionary for your own field names.

In [ ]:
from grat.loaders.get_sites import load_sites

ATTRIBUTE_RENAMER = {
    "GRATID": "grat_id",
    "R_TYPE": "recharge_type",
    "ACRES": "acres",
    "CROP_OFR": "crop",
    "CROP_IL": "in_lieu_crop",
    "WAFRSOURCE": "wafr_sources_primary",
    "WAFRSOUR_1": "wafr_sources_secondary",
    "BOTTLENECK": "bottleneck_ids",
    "SAGBI_CLAS": "soil_type",
    "PRECIP_IDX": "precip_grid",
    "BASIN_CAP": "max_capacity",
    "BASIN_PERC": "perc_rate",
    "CFS_LIMIT": "cfs_limit",
    "RSI": "rsi",
    "RMA": "rma",
    "ET_Region": "region_number",
    "DISTRICT": "district",
    "SUBBASIN": "subbasin",
    "WATERSHED": "watershed",
    "AGENCY": "agency",
    "NONDISTRIC": "nondistrict",
    "PROJECTS": "projects",
}

site_rows = load_sites(OAK_VALLEY / "oakvalley.shp", attribute_renamer=ATTRIBUTE_RENAMER)
len(site_rows), site_rows[0]

Canals, bottlenecks, WAFR, precipitation and the crop compatibility calendars each come from their own csv.

In [ ]:
from grat.loaders.get_bottlenecks import load_bottlenecks
from grat.loaders.get_canals import load_canals
from grat.loaders.get_ccc import get_crop_calendar, get_il_crop_calendar
from grat.loaders.get_precip import get_precip
from grat.loaders.get_wafr import load_wafr

canals = load_canals(OAK_VALLEY / "canals.csv")
bottleneck_rows = pd.read_csv(OAK_VALLEY / "bottlenecks.csv").to_dict("records")
bottlenecks = load_bottlenecks(OAK_VALLEY / "bottlenecks.csv")
wafr = load_wafr(OAK_VALLEY / "WAFR_optimistic.csv")
precip = get_precip(OAK_VALLEY / "Precip_optimistic.csv")
crop_calendar = get_crop_calendar(OAK_VALLEY / "CCC_ofr.csv")
crop_calendar_in_lieu = get_il_crop_calendar(OAK_VALLEY / "CCC_inlieu.csv")

len(canals), len(bottlenecks), sorted(wafr[2000])

## 2. Check the data before running the model

`validate_grat_inputs` runs data validation checks - missing or duplicate ids, non-numeric or negative values, crop/soil/WAFR-source/bottleneck references that don't match the other inputs, inconsistent day counts.

In [ ]:
from grat.validation import validate_grat_inputs

report = validate_grat_inputs(
    site_rows,
    wafr,
    bottleneck_rows=bottleneck_rows,
    crop_calendar=crop_calendar,
    crop_calendar_in_lieu=crop_calendar_in_lieu,
    precip=precip,
)
report

`report.ok` is `True` here - no errors - but the warnings are real: a few `in_lieu_crop` values in the sample data (e.g. `"Wheat"`) aren't in the in-lieu crop calendar, and two bottleneck ids referenced by sites aren't in `bottlenecks.csv`. Neither stops the model from running - those sites are simply excluded from in-lieu recharge, or treated as having no bottleneck - but they're worth knowing about before trusting the results. `report.errors` and `report.warnings` give the issues as a list if you want to act on them in code rather than just reading the summary:

In [ ]:
report.warnings

## 3. Build the model's site objects

`build_sites` turns the raw dictionaries into `Farm`/`Basin` objects; `drop_invalid_sites` removes the ones whose crop, in-lieu crop or soil type don't match the crop compatibility calendar (the same conditions the validation report's warnings pointed at). Canals come pre-built from `load_canals`, so they join the list separately.

In [ ]:
from grat.loaders.get_ccc import ccc_valid_crops, ccc_valid_soil_types
from grat.loaders.get_sites import build_sites, drop_invalid_sites

farms_and_basins = build_sites(site_rows)
farms_and_basins = drop_invalid_sites(
    farms_and_basins,
    valid_crop_list=ccc_valid_crops(crop_calendar),
    valid_il_crop_list=ccc_valid_crops(crop_calendar_in_lieu),
    valid_soil_types=ccc_valid_soil_types(crop_calendar),
)

sites = canals + farms_and_basins
len(sites)

## 4. Run the model and produce reports

`run_grat` drives the model across a span of water years and writes the requested reports to an output folder (zipped by default). The sample data covers water years 1969-2018; this runs a 3-year slice so the notebook finishes quickly - drop `years` entirely (or widen it) to run the full period, which takes well under a minute for Oak Valley's ~650 sites.

`grat.reports.catalog.ALL_REPORTS` lists every report name that can be requested.

In [ ]:
import tempfile

from grat import run_grat

result = run_grat(
    sites,
    wafr,
    bottlenecks,
    crop_calendar,
    precip,
    crop_calendar_in_lieu=crop_calendar_in_lieu,
    years=range(2010, 2013),
    reports=["annual_summary_recharge", "acres_available_and_used", "yearly_recharge_volumes"],
    output_parent=tempfile.mkdtemp(prefix="grat_oak_valley_"),
)

result["reports"], result["zip_path"]

## 5. Read a report back

Anything `run_grat` wrote to csv can be read back as a dataframe with `read_report`, without re-running the model.

In [ ]:
from grat import read_report

read_report(result["output_folder"], "annual_summary_recharge")

`result["series"]` holds the reports meant for charts rather than csv - requested the same way, by name, but returned as plain dicts.

In [ ]:
pd.DataFrame(result["series"]["yearly_recharge_volumes"]["Years"]).T